# Lab 8: Hypothesis Testing — Is Job Training Worth It?
## ECON 3916: Applied Machine Learning for Economists
### Guided Construction Lab | 40 min Core + 10 min Write it yourself + 15 min Extension + take-home Challenge

---

**Learning Objectives:**
- Conduct one-sample and two-sample t-tests in Python and interpret the results
- Compute Cohen's d to assess practical significance beyond statistical significance
- Perform a priori power analysis to determine required sample sizes
- Apply Bonferroni and Benjamini-Hochberg corrections for multiple comparisons
- Write Welch's t-statistic as a function of means, SDs and sample sizes, and check it against scipy

**Dataset:** LaLonde National Supported Work (NSW) Demonstration — treatment vs. control group earnings

**Prerequisites:** the Part 0 sections of Labs 1-5, and the labs before this one.

**How this lab works:** Parts 1-3 are GUIDED: run each cell as it is, then answer the questions. Part 4 runs one guided step, then has two short blanks to fill in. "Write it yourself" is required: a short function you write from a blank cell. The Extension is optional, and the Challenge is a take-home.

**AI in this lab:** you may ask Colab to *explain* an error or a line of code (click **Explain error**, or select a line and ask Gemini to explain it). Do not ask it to write or fix code, and turn off AI code completion. This lab has no AI coding section: you write every line yourself. The one use of AI is drafting your README from the prompt at the end.

---

> ### What you write, and what you run
>
> **You write:**
> - **Part 4, Step 11:** two blanks (`____`), one line each: the Bonferroni correction and the Benjamini-Hochberg correction.
> - **Write it yourself (required),** after Part 4: `welch_t()`, a function that computes Welch's t-statistic from two groups' means, SDs and sizes, then one call to it and a print. About six lines from a blank cell. The check cell below it must print "Passed".
> - The answers to the interpretation questions after each part, in text cells.
>
> **You run and read:** everything else. Setup, Parts 1-3, Step 10, and the Extension are working code to learn from. Read each cell before you run it, and make sure you can say what each line does. The Extension (optional) asks you to change one number, run the cell again, and answer in one sentence.
>
> **The Challenge** (take-home) has three one-line blanks in a given simulation, and a written reflection. The **README** is text only: the prompt at the end is the one place you use AI, and only for documentation.

## Setup

In [ ]:
# GUIDED — run as-is
# Step 1: install pingouin, then import libraries
!pip install -q pingouin statsmodels

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import pingouin as pg
from statsmodels.stats.power import TTestIndPower
from statsmodels.stats.multitest import multipletests
from pathlib import Path

# savefig will not create a folder, and a fresh Colab session has no figures/ folder
Path("figures").mkdir(exist_ok=True)

print("Setup complete.")

## Part 1: Load and Explore the LaLonde NSW Data (GUIDED — 10 min)

The National Supported Work (NSW) Demonstration was a 1970s randomized job training program
for disadvantaged workers. Robert LaLonde's 1986 analysis of this data became one of the
most replicated studies in economics. The central question: **did the training increase earnings?**

In [ ]:
# GUIDED — run as-is
# Step 2: load the NSW experimental sample

# Dehejia & Wahba (2002) version of LaLonde's data, served by NBER; .dta is a Stata file
url = "https://users.nber.org/~rdehejia/data/nsw_dw.dta"
df = pd.read_stata(url)

print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
# treat is 1 for a trainee and 0 for a control; True counts as 1 in a sum, so each sum counts rows
print(f"\nTreatment group: {(df['treat'] == 1).sum()} treated, {(df['treat'] == 0).sum()} control")
print("\nFirst 5 rows:")
df.head()

In [ ]:
# GUIDED — run as-is
# Step 3: summary statistics by treatment status

# re78 is earnings in 1978, the outcome; .values turns a column into a plain NumPy array
treated = df[df["treat"] == 1]["re78"].values
control = df[df["treat"] == 0]["re78"].values

# ,.2f prints commas between thousands and 2 decimals.
# ddof=1 divides by n - 1, which gives the sample standard deviation; plain np.std divides by n
print("1978 earnings")
print(f"Treatment group (n={len(treated)}):")
print(f"  Mean:   ${np.mean(treated):,.2f}")
print(f"  Median: ${np.median(treated):,.2f}")
print(f"  Std:    ${np.std(treated, ddof=1):,.2f}")
print(f"\nControl group (n={len(control)}):")
print(f"  Mean:   ${np.mean(control):,.2f}")
print(f"  Median: ${np.median(control):,.2f}")
print(f"  Std:    ${np.std(control, ddof=1):,.2f}")
print(f"\nRaw difference in means: ${np.mean(treated) - np.mean(control):,.2f}")

**Expected output:** 185 trainees with mean 1978 earnings of \$6,349.14, and 260 controls with
\$4,554.80: a raw difference of \$1,794.34. In both groups the median is well below the mean
(\$4,232.31 and \$3,138.80), the first sign that the earnings are skewed.

In [ ]:
# GUIDED — run as-is
# Step 4: the two earnings distributions, side by side

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# alpha=0.6 makes the bars see-through, so the overlap between the groups shows
axes[0].hist(treated, bins=30, alpha=0.6, color="steelblue", label=f"Treatment (n={len(treated)})")
axes[0].hist(control, bins=30, alpha=0.6, color="peru", label=f"Control (n={len(control)})")
axes[0].axvline(np.mean(treated), color="navy", linestyle="--", label=f"Treat mean: ${np.mean(treated):,.0f}")
axes[0].axvline(np.mean(control), color="saddlebrown", linestyle="--", label=f"Control mean: ${np.mean(control):,.0f}")
axes[0].set_xlabel("1978 Earnings ($)")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Earnings Distribution by Treatment Status")
axes[0].legend()

axes[1].boxplot([treated, control], tick_labels=["Treatment", "Control"])
axes[1].set_ylabel("1978 Earnings ($)")
axes[1].set_title("Earnings Box Plot by Treatment Status")

plt.tight_layout()
plt.savefig("figures/ch08_lalonde_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

**What to look for:** both distributions are right-skewed, with a tall bar at zero and a long tail
of high earners. A t-test compares means, not whole distributions. It does not need the earnings
themselves to be normal: it needs the *sample mean* to be close to normal, and with 185 and 260
people per group the Central Limit Theorem (Chapter 5) makes that a fair bet, even for skewed data.

### Interpretation Questions

1. Why are both earnings distributions right-skewed? Is this typical for economic data?
2. Looking at the histograms, do you think the treatment increased earnings? How confident are you?
3. Why might comparing means be misleading here? What does the box plot reveal that the histogram does not?

*Your answers here:*

1. 
2. 
3. 

## Part 2: One-Sample and Two-Sample T-Tests (GUIDED — 10 min)

We will conduct two hypothesis tests:
- **One-sample:** Is the average control group earnings different from $5,000?
- **Two-sample (Welch's):** Is there a significant difference between treatment and control earnings?

<!-- idiom-note -->
> ### New idiom — the one-line `if`/`else`, and comprehensions
>
> A **ternary** is Python's one-line `if`/`else`. Unlike an ordinary `if`, it
> produces a *value*, so it can sit inside an f-string:
>
> ```python
> f"Decision: {'Reject H0' if p < 0.05 else 'Fail to reject H0'}"
> ```
>
> Read it in the order it is written: *the first thing, if the condition holds,
> otherwise the second thing.* They chain, which is how the effect-size cell
> below produces three bands:
>
> ```python
> "Small" if d < 0.5 else "Medium" if d < 0.8 else "Large"
> ```
>
> You will also see a **list comprehension** in this lab — a new list built
> from an old one in one line:
>
> ```python
> [power_for(n) for n in sample_sizes]      # "power_for(n), for every n"
> ```
>
> Anything a comprehension does could be a `for` loop with `.append()`; it is
> just shorter. The two combine — `["high" if x > 5 else "low" for x in rates]`
> — which looks cryptic exactly once. `np.where` from Lab 5 does that same job
> on a whole array and is faster; use the comprehension on plain lists.


In [ ]:
# GUIDED — run as-is
# Step 5: one-sample t-test
# H0: mean control earnings = $5,000.   H1: mean control earnings != $5,000

hypothesized_mean = 5000
t_stat_1, p_value_1 = stats.ttest_1samp(control, popmean=hypothesized_mean)

# :, in {hypothesized_mean:,} adds the thousands comma
print("One-sample t-test")
print(f"H0: Average control group earnings = ${hypothesized_mean:,}")
print(f"H1: Average control group earnings != ${hypothesized_mean:,}")
print(f"\nSample mean:  ${np.mean(control):,.2f}")
print(f"T-statistic:  {t_stat_1:.4f}")
print(f"P-value:      {p_value_1:.4f}")
# The one-line if/else from the note above
print(f"\nDecision at alpha=0.05: {'Reject H0' if p_value_1 < 0.05 else 'Fail to reject H0'}")

**Expected output:** the control mean is \$4,554.80, and t = −1.3091, p = 0.1917: fail to reject
H0. There is not enough evidence that average control earnings differ from \$5,000. That is not
the same as showing they *are* \$5,000; it says the data cannot rule \$5,000 out.

In [ ]:
# GUIDED — run as-is
# Step 6: two-sample t-test, Welch's version, which does not assume equal variances
# H0: mu_treatment - mu_control = 0.   H1: mu_treatment - mu_control != 0

# equal_var=False picks Welch's test. treated comes first, so a positive t means trainees earned more
t_stat_2, p_value_2 = stats.ttest_ind(
    treated, control,
    equal_var=False,
    alternative="two-sided"
)

mean_diff = np.mean(treated) - np.mean(control)

print("Two-sample Welch's t-test")
print(f"H0: Treatment earnings = Control earnings (no training effect)")
print(f"H1: Treatment earnings != Control earnings")
print(f"\nTreatment mean: ${np.mean(treated):,.2f}")
print(f"Control mean:   ${np.mean(control):,.2f}")
print(f"Difference:     ${mean_diff:,.2f}")
print(f"\nT-statistic:  {t_stat_2:.4f}")
print(f"P-value:      {p_value_2:.4f}")
print(f"\nDecision at alpha=0.05: {'Reject H0' if p_value_2 < 0.05 else 'Fail to reject H0'}")

**Expected output:** a difference of \$1,794.34, with t = 2.6741 and p = 0.0079: reject H0 at
alpha = 0.05. You will compute that 2.6741 yourself, by hand, in "Write it yourself" after Part 4.

### Interpretation Questions

1. For the one-sample test: if we had used a hypothesized mean of $4,000 instead of $5,000, would you expect the p-value to be larger or smaller? Why?
2. For the two-sample test: we used `equal_var=False` (Welch's). What would go wrong if we assumed equal variances when they are not equal?
3. The two-sample test tells us the difference is "statistically significant." Does that mean the job training program is worth funding? What additional information would you need?

*Your answers here:*

1. 
2. 
3. 

## Part 3: Effect Size and Power Analysis (GUIDED — 10 min)

A p-value tells you whether a difference is hard to explain by chance alone. Cohen's d tells
you how big the difference is. Power analysis tells you whether your test could have detected
an effect of that size in the first place.

In [ ]:
# GUIDED — run as-is
# Step 7: Cohen's d, the difference in means measured in standard deviations

# By hand: pool the two SDs into one, weighting each group by its n - 1
n1, n2 = len(treated), len(control)
s1, s2 = np.std(treated, ddof=1), np.std(control, ddof=1)
s_pooled = np.sqrt(((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2))
d_manual = (np.mean(treated) - np.mean(control)) / s_pooled

print("Effect size: Cohen's d")
print(f"Pooled std:  ${s_pooled:,.2f}")
print(f"Cohen's d:   {d_manual:.4f}")

# The same number from pingouin, as a check on the hand calculation
d_pg = pg.compute_effsize(treated, control, paired=False, eftype="cohen")
print(f"Cohen's d (pingouin): {d_pg:.4f}")

print("\nBenchmarks: small d = 0.2, medium d = 0.5, large d = 0.8")
# abs() drops the sign; the chained if/else from the note above picks the band
print(f"Our d = {d_manual:.2f} -> {'Small' if abs(d_manual) < 0.5 else 'Medium' if abs(d_manual) < 0.8 else 'Large'} effect")
print(f"The ${mean_diff:,.0f} earnings difference represents {abs(d_manual):.2f} standard deviations.")

**Expected output:** a pooled SD of \$6,579.54 and d = 0.2727, the same by hand and from
pingouin. That is a small effect on Cohen's scale: the \$1,794 gap is about a quarter of a
standard deviation. Whether \$1,794 a year is worth a training programme is a question about
dollars and costs, which d cannot answer.

In [ ]:
# GUIDED — run as-is
# Step 8: power analysis, for our study and for a planned one

power_analysis = TTestIndPower()

# Power: the chance the test detects an effect of size d, with our two group sizes.
# ratio is the second group's size divided by the first's
achieved_power = power_analysis.solve_power(
    effect_size=abs(d_manual),
    nobs1=n1,
    alpha=0.05,
    ratio=n2 / n1,
    alternative="two-sided"
)
print("Power of our study")
print(f"Effect size (d):     {abs(d_manual):.3f}")
print(f"Treatment n:         {n1}")
print(f"Control n:           {n2}")
print(f"Achieved power:      {achieved_power:.3f} ({achieved_power*100:.1f}%)")
print(f"Target:              0.800 (80%)")

# solve_power solves for the one argument you leave out: here nobs1, the n per group
n_required_03 = power_analysis.solve_power(
    effect_size=0.3,
    alpha=0.05,
    power=0.80,
    ratio=1.0,
    alternative="two-sided"
)
print("\nRequired sample size")
print(f"To detect d=0.3 (small-medium) with 80% power:")
print(f"  n per group = {n_required_03:.0f}")
print(f"  Total participants = {2 * n_required_03:.0f}")

**Expected output:** achieved power of 0.808, just above the 80% target. It was computed at the
effect size we *observed*, so it describes this sample rather than the design (Question 1 below
comes back to this). To detect d = 0.3 with 80% power takes 175 per group. The total, 351, is not
2 × 175 because the code doubles the unrounded answer, about 175.4, before rounding. In practice
you round up, to 176 per group.

In [ ]:
# GUIDED — run as-is
# Step 9: power curves, the power for each n per group, at five effect sizes

fig, ax = plt.subplots(figsize=(10, 6))
sample_sizes = np.arange(10, 500)      # 10, 11, ..., 499 per group

# Each tuple is one curve: its effect size, its legend label, and its colour
for d, label, color in [
    (0.2, "Small (d=0.2)", "peru"),
    (abs(d_manual), f"Our effect (d={abs(d_manual):.2f})", "navy"),
    (0.3, "d=0.3", "lightsteelblue"),
    (0.5, "Medium (d=0.5)", "steelblue"),
    (0.8, "Large (d=0.8)", "seagreen"),
]:
    # The list comprehension from the note above: one power for every n in sample_sizes
    powers = [power_analysis.solve_power(
        effect_size=d, nobs1=n, alpha=0.05,
        ratio=1.0, alternative="two-sided"
    ) for n in sample_sizes]
    ax.plot(sample_sizes, powers, label=label, linewidth=2, color=color)

ax.axhline(y=0.80, color="gray", linestyle="--", label="80% power target")
ax.axvline(x=n1, color="red", linestyle=":", label=f"Our n={n1}")
ax.set_xlabel("Sample Size per Group")
ax.set_ylabel("Statistical Power")
ax.set_title("Power Curves — How Sample Size Determines Detectable Effects")
ax.legend(loc="lower right")
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig("figures/ch08_power_curves.png", dpi=150, bbox_inches="tight")
plt.show()

**Reading the chart.** Our effect is d = 0.27, not 0.3, so it gets its own curve. The d = 0.3
curve crosses 80% at the 175 per group from Step 8; our own curve needs about 212. Smaller effects
need much larger samples: halving the effect size roughly quadruples the n you need.

Every curve assumes two groups of the same size. Ours are 185 and 260, so where the red line meets
our curve (about 74%) is slightly below the 0.808 that Step 8 computed for the real,
unequal groups.

### Interpretation Questions

1. Read the achieved power printed in Step 8. Was it above or below 80%, and what does that mean for our confidence in the two-sample t-test result? (Note that it was computed at the effect size we *observed*, not one fixed in advance — that is post-hoc power, and it describes this sample rather than the design.)
2. If a new job-training study wanted to detect a d=0.2 effect with 80% power, approximately how many participants per group would it need? Read this from the power curve.
3. A colleague says: "My t-test gave p=0.12, so the treatment doesn't work." What is wrong with this reasoning? How would you advise them?

*Your answers here:*

1. 
2. 
3. 

## Part 4: Multiple Comparisons (YOUR TASK — 10 min)

What if we test the treatment effect across multiple demographic subgroups?
With enough tests, a "significant" result by chance alone becomes close to certain.

**One thing to notice about the five subgroups below: they are not disjoint.**
"No HS Diploma" and "HS Diploma+" are complements — between them they cover
every row in the sample — and both of them overlap "Black" and "Age < 25".
So these are five tests on heavily overlapping slices of the same 445 people,
not five independent experiments.

That matters differently for the two corrections. Bonferroni makes *no*
independence assumption, so it still controls the family-wise error rate here.
Benjamini-Hochberg's FDR guarantee holds under independence or positive
dependence; with overlapping subgroups you are relying on the second of those.
Neither correction, though, can tell you whether five was the right number of
tests to count — that is a decision you make before you look at the data.

In [ ]:
# GUIDED — run as-is
# Step 10: the treatment effect in five demographic subgroups

# Each entry is a True/False mask, True for the rows in that subgroup
subgroups = {
    "Black": df["black"] == 1,
    "Hispanic": df["hispanic"] == 1,
    "No HS Diploma": df["education"] < 12,
    "HS Diploma+": df["education"] >= 12,
    "Age < 25": df["age"] < 25,
}

results = []
print("Subgroup t-tests (uncorrected)")
# <18 pads a name to 18 characters; >8 right-aligns in 8
print(f"{'Subgroup':<18} {'n_treat':>8} {'n_ctrl':>8} {'Diff ($)':>10} {'t-stat':>8} {'p-value':>10}")
print("-" * 65)

# .items() gives each subgroup's name and its mask together
for name, mask in subgroups.items():
    sub = df[mask]
    t_grp = sub[sub["treat"] == 1]["re78"].values
    c_grp = sub[sub["treat"] == 0]["re78"].values
    t_s, p_s = stats.ttest_ind(t_grp, c_grp, equal_var=False)
    diff = np.mean(t_grp) - np.mean(c_grp)
    results.append({"subgroup": name, "p_value": p_s, "t_stat": t_s, "diff": diff})
    sig = "*" if p_s < 0.05 else ""        # a star marks p < 0.05
    print(f"{name:<18} {len(t_grp):>8} {len(c_grp):>8} {diff:>10,.0f} {t_s:>8.3f} {p_s:>10.4f} {sig}")

raw_pvals = [r["p_value"] for r in results]
# True counts as 1, so this counts the p-values below 0.05
raw_sig = sum(p < 0.05 for p in raw_pvals)
print(f"\nUncorrected: {raw_sig} of {len(raw_pvals)} subgroups are 'significant' at alpha=0.05")

**Expected output:** two of the five subgroups are significant before any correction: Black
(p = 0.0073) and HS Diploma+ (p = 0.0302). Hispanic has only 11 trainees and 28 controls.

In [ ]:
# YOUR TASK — fill in the two blanks (____)
# Step 11: Bonferroni and Benjamini-Hochberg corrections

p_values = np.array(raw_pvals)

# multipletests (imported in Setup) takes the p-values, an alpha and a method name, and gives
# back four things: which tests to reject, the adjusted p-values, and two we skip with _.
# Chapter 8, Section 8.10 shows how it is called. Its method names are listed in help(multipletests).

# Blank 1: the Bonferroni correction, at alpha = 0.05
reject_bonf, pvals_bonf, _, _ = ____

# Blank 2: the Benjamini-Hochberg (false discovery rate) correction, at alpha = 0.05
reject_bh, pvals_bh, _, _ = ____

print("Multiple comparisons correction")
print(f"{'Subgroup':<18} {'Raw p':>8} {'Bonf p':>10} {'BH p':>10} {'Bonf':>6} {'BH':>6}")
print("-" * 60)
# enumerate counts as it loops: i is 0, 1, 2, ... beside each result
for i, r in enumerate(results):
    sig_b = "Sig" if reject_bonf[i] else ""
    sig_bh = "Sig" if reject_bh[i] else ""
    print(f"{r['subgroup']:<18} {p_values[i]:>8.4f} {pvals_bonf[i]:>10.4f} {pvals_bh[i]:>10.4f} {sig_b:>6} {sig_bh:>6}")

print(f"\nBonferroni significant: {np.sum(reject_bonf)}")
print(f"BH (FDR) significant:   {np.sum(reject_bh)}")

**Expected output:** one subgroup survives each correction, and it is the same one: Black, whose
p of 0.0073 becomes 0.0367 under both. HS Diploma+ does not survive: 0.1512 under Bonferroni and
0.0756 under BH. *NameError: name '____' is not defined* means a blank is still empty.

### Interpretation Questions

1. How many subgroup effects survived Bonferroni correction? How many survived BH?
2. Why is Bonferroni more conservative than BH? What does each one control?
3. If you were a policy maker deciding whether to fund this job training program for a *specific* demographic group based on these results, would you trust the uncorrected or corrected p-values? Why?
4. "No HS Diploma" and "HS Diploma+" split the sample in two. If you dropped one of them, the family would be four tests instead of five and every Bonferroni-adjusted p-value would fall. Does that make the four-test version more honest, or less? What stops this from being a way to manufacture significance?

*Your answers here:*

1. 
2. 
3.
4. 


---

## Write it yourself (required — 10 min)

This part is required, and it is short. The check cell below it must print "Passed" before you
submit.

**The task.** In Step 6, scipy took the two arrays of earnings and gave back t = 2.6741. Now
compute that number yourself, from six summary numbers. Write a function
`welch_t(mean_1, sd_1, n_1, mean_2, sd_2, n_2)` that takes each group's mean, standard deviation
and size, in that order, and gives back Welch's t-statistic (Chapter 8, Section 8.5):

$$t = \frac{\bar{x}_1 - \bar{x}_2}{\sqrt{\dfrac{s_1^2}{n_1} + \dfrac{s_2^2}{n_2}}}$$

Here $\bar{x}$ is a group's mean, $s$ its standard deviation and $n$ its size. Then call your
function on Step 3's numbers, with the treatment group as group 1 and the control group as
group 2. Store the result as `t_by_hand`, and print it beside Step 6's `t_stat_2`. Do not call scipy
inside `welch_t`.

- It takes about six lines: the `def` line, a docstring, a line or two inside, the call, and a print.
- `x**2` squares a number, and `np.sqrt(x)` takes a square root.
- $s^2$ is the variance, the square of the SD. Step 3 computed each SD with `ddof=1`; use the same.
- The bottom of the fraction is the standard error of the difference: Lab 7's $s/\sqrt{n}$, for
  two groups at once. Each group keeps its own variance. Step 7's `s_pooled` belongs to Cohen's d,
  not to Welch's t.
- Lab 6's `split_by_share()` shows the shape of a function: the `def` line, a one-line docstring,
  the body, and `return`.

Then run the check cell. It calls your function on Step 3's numbers and compares the answer with
Step 6. It swaps the two groups, which should flip the sign. It runs your function on each of the
five subgroups from Step 10 and compares with the t-statistics there, which scipy computed. Last,
it checks `t_by_hand`. `assert` stops with its message when its condition is False; the message
tells you what to fix.

In [ ]:
# WRITE IT YOURSELF (required): welch_t(), then t_by_hand from Step 3's numbers

In [ ]:
# CHECK — for "Write it yourself" above

def agree(a, b):
    """True when two numbers match to 4 decimal places, the precision the lab prints."""
    return abs(a - b) < 0.0001


# globals() holds every name defined so far in this notebook
if "welch_t" not in globals():
    print("Not written yet: write welch_t() in the cell above and run it, then run this check.")
else:
    # Step 3's six numbers; float() turns each into a plain Python number
    mean_t, sd_t, n_t = float(np.mean(treated)), float(np.std(treated, ddof=1)), len(treated)
    mean_c, sd_c, n_c = float(np.mean(control)), float(np.std(control, ddof=1)), len(control)
    yours = welch_t(mean_t, sd_t, n_t, mean_c, sd_c, n_c)
    assert yours is not None, "welch_t gave back nothing. Does it end with a return line?"

    # What four common slips would give. scipy computes each one, from altered inputs
    pooled = stats.ttest_ind_from_stats(mean_t, sd_t, n_t, mean_c, sd_c, n_c, equal_var=True).statistic
    n_minus_1 = stats.ttest_ind_from_stats(mean_t, sd_t, n_t - 1, mean_c, sd_c, n_c - 1, equal_var=False).statistic
    # An SD left unsquared is the variance of a group whose SD is its square root
    sd_unsquared = stats.ttest_ind_from_stats(mean_t, np.sqrt(sd_t), n_t, mean_c, np.sqrt(sd_c), n_c, equal_var=False).statistic
    # Dividing by the sum itself, not its square root, gives t squared over the difference
    no_root = t_stat_2**2 / (mean_t - mean_c)

    assert not agree(yours, -t_stat_2), (
        "Right size, wrong sign. Subtract group 2's mean from group 1's: treated minus control, as in Step 6.")
    assert not agree(yours, pooled), (
        f"{yours:.4f} is the pooled t, which assumes equal variances. Step 6 used Welch's: each "
        "group's own variance over its own n. Step 7's s_pooled is for Cohen's d only.")
    assert not agree(yours, n_minus_1), (
        f"{yours:.4f} is a little too small: divide each variance by n, not n - 1. "
        "The n - 1 is already inside each SD (ddof=1).")
    assert not agree(yours, sd_unsquared), (
        f"{yours:.4f} is far too large: square each SD. The formula needs the variance, sd**2.")
    assert not agree(yours, no_root), (
        f"{yours:.4f} is far too small: take the square root of the sum at the bottom. "
        "Without it you divide by a variance, not a standard error.")
    assert abs(yours) < 100, f"{yours:.1f} is in the thousands: put mean_1 - mean_2 in parentheses before dividing."
    assert agree(yours, t_stat_2), (
        f"welch_t gives {yours:.4f} for Step 3's numbers, and Step 6 gives {t_stat_2:.4f}. Check the "
        "argument order (mean, SD, n for group 1, then for group 2) and the formula in the task.")

    # Swap the groups: the difference, and so t, should turn negative
    assert agree(welch_t(mean_c, sd_c, n_c, mean_t, sd_t, n_t), -t_stat_2), (
        "With the groups swapped, t should be negative, and yours is not. Did you use abs()? "
        "Keep the sign: it says which group earned more.")

    # The five subgroups from Step 10, where scipy computed each t-statistic
    for r in results:
        rows = df[subgroups[r["subgroup"]]]
        g1 = rows[rows["treat"] == 1]["re78"].values
        g2 = rows[rows["treat"] == 0]["re78"].values
        t_sub = welch_t(float(np.mean(g1)), float(np.std(g1, ddof=1)), len(g1),
                        float(np.mean(g2)), float(np.std(g2, ddof=1)), len(g2))
        assert agree(t_sub, r["t_stat"]), (
            f"Right on the full sample, but for {r['subgroup']} you get {t_sub:.4f} and Step 10 has "
            f"{r['t_stat']:.4f}. Does welch_t use its own arguments, or names from earlier cells like n1 or s1?")

    assert "t_by_hand" in globals(), (
        "Your function works. Now call it on Step 3's numbers and store the result as t_by_hand.")
    # The same call with SDs that divide by n (np.std's default), to spot a missing ddof=1
    ddof_0 = welch_t(mean_t, float(np.std(treated)), n_t, mean_c, float(np.std(control)), n_c)
    assert not agree(t_by_hand, ddof_0), (
        "t_by_hand is a little too large: its SDs divide by n. Use np.std(..., ddof=1), as Step 3 does.")
    assert agree(t_by_hand, t_stat_2), (
        f"t_by_hand is {t_by_hand:.4f}, but Step 6 gives {t_stat_2:.4f}. "
        "Is the treatment group first, with Step 3's means, SDs and sizes?")
    print(f"Passed. Your t: {t_by_hand:.4f}. scipy's in Step 6: {t_stat_2:.4f}.")
    print(f"All {len(results)} subgroup t-statistics from Step 10 match too.")

---

## Extension (Optional — 15 min): The P-Value Distribution Under the Null

If the null hypothesis is true, p-values should be *uniformly distributed* on [0, 1].
Let's verify this with simulation — and see what happens when the null is false.

In [ ]:
# EXTENSION — run as-is
# Simulate p-values when the null is true, and when it is false

np.random.seed(42)          # the same random draws on every run
n_simulations = 2000
n_per_group = 50

# Simulation 1: both groups come from the SAME distribution, so the null is true.
# _ stands for a value we never use: the loop counter, and the t-statistic
p_values_null = []
for _ in range(n_simulations):
    group_a = np.random.normal(loc=100, scale=15, size=n_per_group)
    group_b = np.random.normal(loc=100, scale=15, size=n_per_group)
    _, p = stats.ttest_ind(group_a, group_b, equal_var=False)
    p_values_null.append(p)

# Simulation 2: group B's mean is 7.5 higher, half of the SD of 15, so d = 0.5 and the null is false
p_values_alt = []
for _ in range(n_simulations):
    group_a = np.random.normal(loc=100, scale=15, size=n_per_group)
    group_b = np.random.normal(loc=107.5, scale=15, size=n_per_group)
    _, p = stats.ttest_ind(group_a, group_b, equal_var=False)
    p_values_alt.append(p)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(p_values_null, bins=20, color="steelblue", edgecolor="white")
# 20 bars share 2,000 p-values, so a flat histogram has 100 in each
axes[0].axhline(y=n_simulations / 20, color="red", linestyle="--", label="Expected if uniform")
axes[0].set_xlabel("P-value")
axes[0].set_ylabel("Frequency")
axes[0].set_title("P-values When Null Is TRUE")
axes[0].legend()

axes[1].hist(p_values_alt, bins=20, color="peru", edgecolor="white")
axes[1].set_xlabel("P-value")
axes[1].set_ylabel("Frequency")
axes[1].set_title("P-values When Null Is FALSE (d=0.5)")

plt.tight_layout()
plt.savefig("figures/ch08_pvalue_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

# p < 0.05 is True or False for each run, so the mean is the share of runs below 0.05
false_positive_rate = np.mean(np.array(p_values_null) < 0.05)
true_positive_rate = np.mean(np.array(p_values_alt) < 0.05)
print(f"False positive rate (null true):   {false_positive_rate:.3f}")
print(f"True positive rate (null false):   {true_positive_rate:.3f}")

**Expected output:** when the null is true, the p-values spread evenly from 0 to 1, and 0.049 of
them fall below 0.05: close to the 5% false positive rate that alpha = 0.05 promises. When the null
is false, the p-values pile up near 0, and 0.712 of them fall below 0.05. That share is the power
of this design: d = 0.5 with 50 per group.

**Try it.** Change `n_per_group` from 50 to 200 and run the cell again. One panel barely moves and
the other changes shape completely. Say which, and say in one sentence what that tells you about
where sample size acts.

*Your answer here:*

---

## Challenge (Take-Home): Simulate the Replication Crisis

A journal that publishes only results with p < 0.05 does not publish 5% false results. How many
of its findings are false depends on how many of the ideas tested were true to begin with, and on
the power of each test. Here you simulate a field of 1,000 studies: 900 test an effect that does
not exist, and 100 test a real effect of d = 0.5, each with 30 people per group. The Extension
told you what share of each kind comes out significant. This asks what share of the significant
results are false.

In [ ]:
# YOUR TASK — take-home Challenge: fill in the three blanks (____)

np.random.seed(42)
n_null = 900            # studies of an effect that does not exist
n_real = 100            # studies of a real effect, d = 0.5
n_per_group = 30

p_null = []
for _ in range(n_null):
    group_a = np.random.normal(0, 1, n_per_group)
    group_b = np.random.normal(0, 1, n_per_group)
    _, p = stats.ttest_ind(group_a, group_b, equal_var=False)
    p_null.append(p)

p_real = []
for _ in range(n_real):
    group_a = np.random.normal(0, 1, n_per_group)
    # Blank 1: group B for a real effect of d = 0.5. The SD is 1, so d is the gap between the means
    group_b = ____
    _, p = stats.ttest_ind(group_a, group_b, equal_var=False)
    p_real.append(p)

# p < 0.05 is True or False for each study, and True counts as 1, so the sum is a count
sig_null = np.sum(np.array(p_null) < 0.05)
# Blank 2: the same count for the studies of a real effect
sig_real = ____
# Blank 3: of all the significant results, the share that came from the null studies
share_false = ____

print(f"Significant, no real effect: {sig_null} of {n_null}")
print(f"Significant, real effect:    {sig_real} of {n_real}")
print(f"Share of significant results that are false: {share_false:.1%}")

**Reflect.** Only about 5% of the null studies came out significant, yet the share of significant
results that are false is far higher than 5%. Explain why, using your two counts. Then say which
would bring the share down more: doubling every study's sample size, or a field where half the
ideas tested are true. You can change `n_per_group`, `n_null` and `n_real` and run the cell again
to check. *NameError: name '____' is not defined* means a blank is still empty.

*Your answer here:*

---
## Digital Portfolio: Your README

Copy the prompt below into Claude or ChatGPT. Fill in each [YOUR VALUE] from your own output
first, and do not quote a number this notebook did not print. **Do NOT ask the AI to write
Python code — only documentation.**

```
I need help writing a project README for my data science lab.
**Important Rule:** Do NOT generate any Python code for me.

**What I did in this lab:**
* Loaded the LaLonde NSW job training experiment (445 people:
  185 trainees and 260 controls)
* Ran a one-sample t-test and a two-sample Welch's t-test on 1978 earnings
* Wrote Welch's t-statistic as a function, welch_t(), and checked it
  against scipy on the full sample and on five subgroups
* Computed Cohen's d, to judge the size of the effect and not only
  whether it is significant
* Ran a power analysis: the power of this study, and the sample size
  needed to detect d = 0.3
* Tested the treatment effect in 5 overlapping demographic subgroups,
  then applied Bonferroni and Benjamini-Hochberg corrections
* Simulated p-value distributions when the null is true and when it is false
* Key finding: trainees earned $[YOUR VALUE] more in 1978 (Welch's
  t = [YOUR VALUE], p = [YOUR VALUE]), a Cohen's d of [YOUR VALUE].
  [YOUR VALUE] of the 5 subgroup effects were significant before
  correction, and [YOUR VALUE] after Bonferroni

**Please write a README.md entry including:**
1. Project Title: Hypothesis Testing — Is Job Training Worth It?
2. Objective: one sentence
3. Methodology: Bullet points of technical steps
4. Key Findings: Summary of results
Write it plainly, in the first person. No words like 'robust', 'comprehensive'
or 'leverage'. Use only the numbers I gave you.
```

---

## Submit this lab on GitHub

**Repository name for this lab:** `econ3916-lab08-hypothesis-testing`

**First time?** Read the **GitHub Setup** page in the Start Here module once,
start to finish. It assumes you have never used git and never seen GitHub.

1. On [github.com](https://github.com): **+** (top right) → **New repository**.
   Name it exactly `econ3916-lab08-hypothesis-testing`, tick **Add a README file**, and create it.
2. In Colab: **File → Save a copy in GitHub**. Choose `econ3916-lab08-hypothesis-testing`, branch `main`,
   commit message `Lab 08 submission`. This pushes the notebook, and the
   notebook's rendered output — charts included — is what is graded.
3. Open the repo on github.com, click `README.md`, then the pencil icon, and
   paste in the README you drafted from the prompt above. **Commit
   changes.** Check every number in it against your own output first: this
   goes out under your name.
4. On Canvas: in Colab use **File → Download → Download .ipynb**, upload that
   file to the assignment, and paste your repository URL in the **Comments**
   box beside the upload. Then click **Submit**. Do not use the Website URL
   tab: Canvas keeps one submission, so a URL sent on its own replaces your
   notebook.

No terminal, no token, nothing to install.